# Eyewear Localization & Brand Attribution — Interactive Streamlit Dashboard
Hosts the Streamlit Web Application on Kaggle GPU and exposes a public HTTPS URL via localtunnel.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/fez-Ox/pxModel-Object-Counting.git'
REPO_DIR = Path('/kaggle/working/pxModel-localization')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())


In [ ]:
# Install PyTorch cu124, transformers, streamlit, easyocr, rapidocr-onnxruntime
print('Installing PyTorch & Vision dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1+cu124', 'torchvision==0.20.1+cu124', '--extra-index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.44.2', 'accelerate', 'timm', 'einops', 'pillow', 'pyyaml', 'streamlit', 'easyocr', 'rapidocr-onnxruntime', 'ftfy'], check=True)
print('Streamlit and pipeline dependencies installed.')


In [ ]:
# Download SAM3 checkpoint
SAM3_CHECKPOINT = REPO_DIR / 'sam3-verbose-counting' / 'checkpoints' / 'sam3.pt'
SAM3_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not SAM3_CHECKPOINT.exists():
    print('Downloading SAM3 checkpoint...')
    from huggingface_hub import hf_hub_download
    hf_token = os.getenv('HF_TOKEN') or os.getenv('HF_ACCESS_TOKEN')
    downloaded_path = hf_hub_download(
        repo_id='facebook/sam3',
        filename='sam3.pt',
        local_dir=str(SAM3_CHECKPOINT.parent),
        token=hf_token,
    )
print('SAM3 Checkpoint ready:', SAM3_CHECKPOINT.exists())


In [ ]:
# Launch Streamlit in background and bridge via localtunnel
print('Starting Streamlit server on port 8501...')
st_proc = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0', '--server.headless', 'true'], cwd=REPO_DIR)
time.sleep(5)

print('Exposing Streamlit via localtunnel...')
lt_proc = subprocess.Popen(['npx', 'localtunnel', '--port', '8501'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Read public URL from localtunnel output
st_url = ''
for _ in range(20):
    line = lt_proc.stdout.readline()
    if 'url is:' in line.lower():
        st_url = line.strip()
        print('
' + '=' * 80)
        print('🚀 STREAMLIT PUBLIC WEB APP URL:', st_url)
        print('=' * 80 + '
')
        break
    time.sleep(1)

# Keep kernel alive to serve Streamlit app
print('Streamlit server active. Serving requests...')
for i in range(120):
    time.sleep(15)
